<a href="https://colab.research.google.com/github/Minimire/CRO-RECOM-AMH-ENG/blob/main/Crop_Recommendation_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files

uploaded = files.upload()

In [ ]:

!pip install -q openpyxl

import pandas as pd

filename = "Crop_recommendation.xlsx"

df = pd.read_excel(filename, engine="openpyxl")

print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nCrop categories:")
print(df["label"].value_counts())

In [ ]:

# ==========================================
# CROP RECOMMENDATION MODEL TRAINING
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Define input features
features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

target = "label"

# 2. Prepare dataset
X = df[features].copy()
y = df[target].copy()

# 3. Encode crop labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 5. Initialize Random Forest
production_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# 6. Train model
production_model.fit(X_train, y_train)

print("\nMODEL TRAINING COMPLETED!")

In [ ]:

# Make predictions
y_pred = production_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("MODEL ACCURACY:")
print(f"{accuracy * 100:.2f}%")

# Classification report
print("\nCLASSIFICATION REPORT:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# Feature importance
importance = pd.DataFrame({
    "Feature": features,
    "Importance": production_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFEATURE IMPORTANCE:")
display(importance)

In [ ]:

import joblib

model_bundle = {
    "production_model": production_model,
    "label_encoder": label_encoder
}

model_filename = "crop_recommendation_system_bundle.pkl"

joblib.dump(model_bundle, model_filename)

print("Model saved successfully!")
print("Filename:", model_filename)

In [ ]:

loaded_bundle = joblib.load(
    "crop_recommendation_system_bundle.pkl"
)

print("Bundle keys:", list(loaded_bundle.keys()))

print(
    "Number of crops:",
    len(loaded_bundle["label_encoder"].classes_)
)

print(
    "Crop names:",
    loaded_bundle["label_encoder"].classes_
)

In [ ]:

from google.colab import files

files.download(
    "crop_recommendation_system_bundle.pkl"
)